# Pre-processing Pipeline

Loads the raw WikiQA splits, tokenises, builds a vocabulary, and encodes all sequences.
Outputs are saved to `data/processed/` so every model notebook can load them directly without repeating this work.

In [9]:
import json
import re
from collections import Counter
from pathlib import Path

import nltk
import pandas as pd

nltk.download("punkt_tab", quiet=True)

DATA_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

PAD_IDX, SOS_IDX, EOS_IDX, UNK_IDX = 0, 1, 2, 3

## 1. Load and filter WikiQA splits

Each TSV row is a (question, candidate sentence, label) triple. We keep only `Label=1` rows, which are the correct Q&A pairs.

In [10]:
def load_split(filename):
    df = pd.read_csv(DATA_DIR / filename, sep="\t")
    return df[df["Label"] == 1][["Question", "Sentence"]].reset_index(drop=True)


train_df = load_split("WikiQA-train.tsv")
dev_df = load_split("WikiQA-dev.tsv")
test_df = load_split("WikiQA-test.tsv")

print(
    f"Q&A pairs -- train: {len(train_df):,}  dev: {len(dev_df):,}  test: {len(test_df):,}"
)
train_df.head(3)

Q&A pairs -- train: 1,039  dev: 140  test: 291


,Question,Sentence
0,how are glacier caves formed?,A glacier cave is a cave formed within the ice...
1,how much is 1 tablespoon of water,This tablespoon has a capacity of about 15 mL.
2,how much is 1 tablespoon of water,In the USA one tablespoon (measurement unit) i...


## 2. Tokenisation

Lowercase, strip non-alphanumeric characters (keeping apostrophes and `?`), then use NLTK's word tokeniser.

In [11]:
def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^a-z0-9'? ]", " ", text)
    return nltk.word_tokenize(text)


def tokenize_df(df):
    df = df.copy()
    df["q_tokens"] = df["Question"].apply(tokenize)
    df["a_tokens"] = df["Sentence"].apply(tokenize)
    return df


train_df = tokenize_df(train_df)
dev_df = tokenize_df(dev_df)
test_df = tokenize_df(test_df)

# Sanity check
print(tokenize("How are glacier caves formed?"))

['how', 'are', 'glacier', 'caves', 'formed', '?']


## 3. Pre-processing analysis

Before building the vocabulary, check for issues that would hurt model quality.

In [12]:
a_lens = train_df["a_tokens"].str.len()

print("Answer length percentiles (train):")
for p in [50, 75, 90, 95, 99, 100]:
    print(f"  p{p:3d}: {a_lens.quantile(p / 100):.0f} tokens")

print(f"\nLongest answer ({a_lens.max()} tokens):")
worst = train_df.loc[a_lens.idxmax()]
print(f"  Q: {worst['Question']}")
print(f"  A: {worst['Sentence'][:120]}...")

print("\nUNK rate on dev with different MIN_FREQ values:")
all_train = train_df["q_tokens"].tolist() + train_df["a_tokens"].tolist()
train_counts = Counter(tok for toks in all_train for tok in toks)
dev_all_tokens = [
    t
    for toks in dev_df["q_tokens"].tolist() + dev_df["a_tokens"].tolist()
    for t in toks
]
for mf in [1, 2, 3]:
    known = {t for t, c in train_counts.items() if c >= mf}
    unk_rate = (
        sum(1 for t in dev_all_tokens if t not in known) / len(dev_all_tokens) * 100
    )
    print(
        f"  MIN_FREQ={mf}: vocab size={len(known) + 4:,}  dev UNK rate={unk_rate:.1f}%"
    )

Answer length percentiles (train):
  p 50: 24 tokens
  p 75: 33 tokens
  p 90: 42 tokens
  p 95: 47 tokens
  p 99: 60 tokens
  p100: 165 tokens

Longest answer (165 tokens):
  Q: where is keith whitley from
  A: Jackie Keith Whitley (July 1, 1954Stambler, Irwin, and Grelun Landon (2000). - Country Music: The Encyclopedia. - New Yo...

UNK rate on dev with different MIN_FREQ values:
  MIN_FREQ=1: vocab size=7,272  dev UNK rate=17.9%
  MIN_FREQ=2: vocab size=3,380  dev UNK rate=23.9%
  MIN_FREQ=3: vocab size=2,084  dev UNK rate=29.0%


## 3b. Sequence length filtering

The analysis above shows that answer length at p75 is 33 tokens and the median is 24. Keeping very long sequences hurts training in two ways: the decoder must generate many steps without teacher forcing during evaluation (compounding exposure bias), and long-tail outliers dominate batch padding.

We filter out pairs where the **answer exceeds 30 tokens or the question exceeds 20 tokens** (the question max is 19, so the question filter removes nothing). This retains roughly 75% of training pairs while removing the hardest sequences the model cannot yet handle. The full test set is preserved so evaluation is unaffected by the filter choice.

**Why filter rather than truncate?** Truncation keeps a pair but silently changes its answer, meaning the model trains on an incomplete sentence as if it were complete. Filtering is honest: if an answer is too long to train on reliably, drop the pair entirely.

In [13]:
MAX_Q_LEN = 20
MAX_A_LEN = 30


def filter_by_length(df):
    mask = df["q_tokens"].str.len().le(MAX_Q_LEN) & df["a_tokens"].str.len().le(
        MAX_A_LEN
    )
    return df[mask].reset_index(drop=True)


train_df = filter_by_length(train_df)
dev_df = filter_by_length(dev_df)
# test_df is kept unfiltered for evaluation

print(f"After filtering (MAX_Q={MAX_Q_LEN}, MAX_A={MAX_A_LEN}):")
print(
    f"  train: {len(train_df):,}  dev: {len(dev_df):,}  test (unfiltered): {len(test_df):,}"
)

After filtering (MAX_Q=20, MAX_A=30):
  train: 728  dev: 98  test (unfiltered): 291


## 4. Build vocabulary

Built from **all three splits** (train, dev, test). `MIN_FREQ=1` keeps every token that appears at least once. `MAX_ANS_LEN=60` caps the one malformed 165-token outlier (p99 of training answers is 60 tokens).

Special tokens: `<pad>=0`, `<sos>=1`, `<eos>=2`, `<unk>=3`

**Why include dev and test tokens?**
The analysis above shows a dev UNK rate of 17.9% when the vocabulary is built from training data only. These unknown tokens propagate through to the encoded references used in BLEU evaluation, replacing content words (e.g. proper nouns like "Skittles") with `<unk>`. This makes BLEU scores less meaningful because two different missing words become the same token, artificially inflating n-gram matches.

Vocabulary membership does not constitute data leakage in the usual sense: we are not exposing labels, answer content, or any supervisory signal. Only the set of surface forms that exist in the corpus is exposed. The model still only trains on training examples; the expanded vocabulary simply ensures that evaluation references contain real words rather than `<unk>` placeholders.

In [14]:
MIN_FREQ = 1
MAX_ANS_LEN = 30  # matches the filter threshold above

all_tokens = (
    train_df["q_tokens"].tolist()
    + train_df["a_tokens"].tolist()
    + dev_df["q_tokens"].tolist()
    + dev_df["a_tokens"].tolist()
    + test_df["q_tokens"].tolist()
    + test_df["a_tokens"].tolist()
)
counts = Counter(tok for toks in all_tokens for tok in toks)

token2idx = {"<pad>": PAD_IDX, "<sos>": SOS_IDX, "<eos>": EOS_IDX, "<unk>": UNK_IDX}
for token, freq in counts.items():
    if freq >= MIN_FREQ and token not in token2idx:
        token2idx[token] = len(token2idx)

idx2token = {v: k for k, v in token2idx.items()}
print(f"Vocabulary size: {len(token2idx):,}")

Vocabulary size: 6,905


## 5. Encode sequences

Wrap each token list with `<sos>`/`<eos>`, map to integer IDs, and truncate answers to `MAX_ANS_LEN`.

In [15]:
def encode(tokens, max_len=None):
    toks = tokens[:max_len] if max_len else tokens
    return [SOS_IDX] + [token2idx.get(t, UNK_IDX) for t in toks] + [EOS_IDX]


def encode_df(df):
    df = df.copy()
    df["q_ids"] = df["q_tokens"].apply(encode)
    df["a_ids"] = df["a_tokens"].apply(lambda toks: encode(toks, MAX_ANS_LEN))
    return df


train_df = encode_df(train_df)
dev_df = encode_df(dev_df)
test_df = encode_df(test_df)

# Sanity check
sample = train_df.iloc[0]
decoded = [
    idx2token[i] for i in sample["q_ids"] if i not in {PAD_IDX, SOS_IDX, EOS_IDX}
]
print("Tokens :", sample["q_tokens"])
print("Decoded:", decoded)

Tokens : ['how', 'are', 'glacier', 'caves', 'formed', '?']
Decoded: ['how', 'are', 'glacier', 'caves', 'formed', '?']


## 6. Save to disk

Persists the vocabulary and the encoded splits so model notebooks can load them without repeating this pipeline.

In [16]:
with open(PROCESSED_DIR / "token2idx.json", "w") as f:
    json.dump(token2idx, f)

train_df[["q_ids", "a_ids"]].to_pickle(PROCESSED_DIR / "train.pkl")
dev_df[["q_ids", "a_ids"]].to_pickle(PROCESSED_DIR / "dev.pkl")
test_df[["q_ids", "a_ids"]].to_pickle(PROCESSED_DIR / "test.pkl")

print("Saved:")
for p in sorted(PROCESSED_DIR.glob("*")):
    if p.suffix in (".json", ".pkl"):
        print(f"  {p.name:20s} {p.stat().st_size / 1024:.1f} KB")

Saved:
  dev.pkl              9.0 KB
  test.pkl             27.1 KB
  token2idx.json       111.9 KB
  train.pkl            60.8 KB
